# RAG(Retrieval-Augmented Generation)
- <font color=red>기존의 LLM을 확장하여 주어진 컨텍스트나 질문에 대해 더욱 정확하고 풍부한 정보를 제공</font>하는 방법
- 즉, 모델이 학습 데이터에 포함되지 않은 외부 데이터를 실시간으로 검색(retrieval)하고 검색된 데이터를 활용(augmented)하여 이를 바탕으로 답변을 생성(generation)하는 것

- 기본 구조
  - <font color=red>검색 단계(Retrieval Phase)</font>: 사용자의 질문이나 컨텍스트를 입력으로 받아서, 이와 관련된 외부 데이터를 검색하는 단계
  - <font color=red>증강 단계(Augmented Phase)</font>: 검색된 데이터를 토큰화, 인코딩, 임베딩 후에 벡터 DB에 저장하고 검색기를 붙이는 단계  
  - <font color=red>생성 단계(Generation Phase)</font>: 벡터 DB에 저장된 데이터와 LLM 모델을 사용하여 사용자의 질문에 답변을 생성하는 단계

- 장점  
  - <font color=red>풍부한 정보 제공</font> : RAG 모델은 검색을 통해 얻은 외부 데이터를 활용하여, 보다 구체적이고 풍부한 정보를 제공
  - <font color=red>실시간 정보 반영</font> : 최신 데이터를 검색하여 반영함으로써, 모델이 실시간으로 변화하는 정보에 대응
  - <font color=red>환각 방지</font> : 검색을 통해 실제 데이터에 기반한 답변을 생성함으로써, 환각 현상이 발생할 위험을 줄이고 정확도 향상

- RAG 8단계 프로세스
  - 사전 준비 단계
    - 문서 가져오기
    - 텍스트 분할
    - 임베딩
    - 벡터 DB에 저장
  - Runtime 단계
    - 검색기 설정
    - 프롬프트 구성
    - LLM 모델 객체 생성
    - 체인 생성 및 실행

# 환경설정

In [ ]:
# Open AI API key 등록
import os

OPENAI_API_KEY='API_KEY'

# 현재 노트북 커널에 환경변수 등록
os.environ['OPENAI_API_KEY']=OPENAI_API_KEY

# 랭스미스로 기록 할 건지 여부
LANGSMITH_TRACING="true"

# 랭스미스 사이트에서 기록 할 나만의 url
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"

# 내 랭스미스 url key
LANGSMITH_API_KEY="API_KEY"

# 내 계정의 프로젝트이름
LANGSMITH_PROJECT="langchain0422"

# 현재 노트북 커널에 환경변수 등록
os.environ['LANGSMITH_TRACING']=LANGSMITH_TRACING
os.environ['LANGSMITH_ENDPOINT']=LANGSMITH_ENDPOINT
os.environ['LANGSMITH_API_KEY']=LANGSMITH_API_KEY
os.environ['LANGSMITH_PROJECT']=LANGSMITH_PROJECT

In [ ]:
# LangChain 라이브러리 설치하기

!pip install langchain langchain-openai langchain-community faiss-cpu pypdf chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180

# RAG 1 : PDF를 학습한 나만의 챗봇 만들기
  - (1) 데이터 로드(Load Data)
    - RAG에 사용할 데이터를 불러오는 단계
  - (2) 텍스트 분할(Text Split)
    - 불러온 데이터를 작은 크기의 단위(chunk)로 분할하는 단계
  - (3) 임베딩 (Embedding) / 인덱싱 (Indexing)
    - 텍스트 데이터를 숫자로 이루어진 벡터로 변환하는 단계
    - 분할된 텍스트를 검색 가능한 형태로 만드는 단계
  - (4) 검색(Retrieval)
    - 사용자의 질문이나 주어진 컨텍스트에 가장 관련된 정보를 찾아내는 단계
  - (5) 생성(Generation)
    - 검색된 정보를 바탕으로 사용자의 질문에 답변을 생성하는 최종 단계

### 0. 기본 LLM 답변 확인하기

In [ ]:
# 라이브러리 불러오기

from langchain.chat_models import init_chat_model   # 모델을 유연하게 생성하는 도구
from langchain_core.output_parsers import StrOutputParser   # 문자열 파싱을 해주는 도구

In [ ]:
# 모델생성
llm_4o_mini = init_chat_model('openai:gpt-4o-mini', max_tokens = 1024)

# 체인구성하기
chain = llm_4o_mini | StrOutputParser()

In [ ]:
chain.invoke('KDT 사업제안을 하려고하는데 사업유형은 어떤게 있어?')

'KDT(한국 데이터 신산업 협회) 사업 제안을 고려할 때, 다음과 같은 여러 유형의 사업을 생각해볼 수 있습니다:\n\n1. **데이터 분석 및 가공 서비스**: 기업이나 기관이 보유한 데이터를 분석하고 가공하여 인사이트를 제공하는 서비스.\n\n2. **AI 및 머신러닝 솔루션 개발**: 인공지능 및 머신러닝 기술을 활용한 맞춤형 솔루션 개발, 예를 들어 예측 모델링, 이미지 인식, 자연어 처리 등.\n\n3. **빅데이터 플랫폼 구축**: 대량의 데이터를 저장, 처리, 분석할 수 있는 플랫폼을 개발하고 운영하는 사업.\n\n4. **데이터 시각화 툴**: 분석된 데이터를 시각적으로 표현할 수 있는 툴이나 소프트웨어 개발.\n\n5. **IoT 데이터 활용**: IoT 기기로부터 수집한 데이터를 분석하고, 이를 기반으로 새로운 서비스를 개발.\n\n6. **데이터 기반 마케팅 전략**: 데이터 분석을 통해 기업의 마케팅 전략을 개선하고 최적화하는 컨설팅 서비스.\n\n7. **데이터 거버넌스 및 개인정보 보호**: 기업이 데이터를 안전하게 관리하고 정부 규정을 준수할 수 있도록 지원하는 서비스.\n\n8. **교육 및 세미나**: 데이터 관련 기술이나 도구에 대한 교육 프로그램이나 세미나 개최.\n\n9. **산업별 데이터 솔루션**: 특정 산업을 대상으로 특화된 데이터 솔루션 제공(예: 헬스케어, 금융, 제조업 등).\n\n10. **결합 데이터 서비스**: 다양한 출처의 데이터를 결합하여 새로운 가치를 창출하는 서비스.\n\n이 외에도 KDT 사업 제안에 적합할 수 있는 다양한 사업 유형이 존재합니다. 해당 사업이 해결하고자 하는 문제와 타겟 시장을 명확히 하여 그에 맞는 사업 유형을 선택하는 것이 중요합니다.'

In [ ]:
chain.invoke('KDT 직종별 단가는 얼마야?')

'KDT(한국디지털터미널)와 관련된 정보는 특정한 직종의 단가를 포함하여 다양한 정보가 포함될 수 있습니다. 그러나 정확한 단가는 직종, 지역, 경력, 스킬셋 등에 따라 다를 수 있습니다. 가장 좋은 방법은 KDT 관련 직종의 공식 웹사이트나 관련된 채용 공고를 직접 확인하거나, 업계의 가이드라인을 참고하는 것입니다.\n\n또한, KDT 관련 산업에 대한 연봉 조사 리포트나 HR 커뮤니티에서의 정보를 통해 평균적인 단가를 파악하는 것도 도움이 될 것입니다. 보다 구체적인 정보를 원하시면 상세한 내용을 제공해 주시면 더 도와드리겠습니다.'

### (1) 데이터 로드(Load Data)
 - RAG에 사용할 데이터를 불러오는 단계

In [ ]:
# 드라이브 마운트

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/딥러닝/LLM활용

/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM활용


In [ ]:
# PDF 파일을 읽어오는 라이브러리

from langchain_community.document_loaders import PyPDFLoader

In [ ]:
# 파일로더 생성
loader = PyPDFLoader('/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM활용/data/KDT.pdf')

# 데이터로딩
documents = loader.load()

In [ ]:
len(documents)

2

In [ ]:
documents[0]

Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2023-12-22T09:41:58+00:00', 'source': '/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM활용/data/KDT.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='- 3 -\n□ K-디지털 트레이닝 아카데미 유형\uf000디지털신기술아카데미:참여기업의훈련수요를기반으로참여기업과훈련기관간협약을체결한후설계된첨단산업·디지털분야훈련과정\uf000벤처\n스타트업아카데미:기업이필요로하는인재를양성하기위해벤처또는스타트업등의기업이속한협\n단체가회원사의인력수요를조사하고훈련기관과협약을체결한후설계된첨단산업·디지털분야훈련과정\uf000지역·산업주도형아카데미:지역별인적자원개발위원회(RSC),산업별인적자원개발위원회(ISC),산업별인적자원개발협의체(SC)가지역·산업인재양성을위해참여기업과훈련기관을발굴,매칭한후3자협약을체결하여설계,운영하는훈련과정\uf000첨단산업디지털선도기업아카데미:첨단산업디지털선도기업이자체적으로또는첨단산업디지털선도기업과행정업무를대행하는운영지원기관이함께운영하는훈련과정\uf000대학주도형아카데미:「고등교육법」상대학또는「산업교육진흥및산학연협력촉진에관한법률」에의한산학협력단이참여기업과협약을체결한후설계된첨단산업·디지털분야훈련과정□ 훈련편성 유형 (※ 유형별 동일·유사한 과정 중복 신청 시 ‘부적합’ 판정)1. 일반 훈련과정(350시간 이상)  ○신기술분야,융합분야등2가지분야에대해서는모든K-디지털트레이닝아카데미유형에서신청가능-다만,기타분야는첨단산업디지털선도기업아카데미와대학주도형아카데미에서만신청가능')

### (2) 텍스트 분할(Text Split)
 - 불러온 데이터를 작은 크기의 단위(chunk)로 분할하는 단계

 - 텍스트 분할(Text Split)
   - <font color=red>CharacterTextSplitter</font> : 주어진 텍스트를 설정한 단위(문단(\n\n), 문장(\n), 단어(빈공백, 형태소) 등)로 분할
   - <font color=red>RecursiveCharacterTextSplitter</font> : 텍스트를 재귀적으로 분할하여 의미적으로 관련 있는 텍스트 조각들이 같이 있도록 하는 목적으로 설계
     - <font color=red>문자 리스트(['\n\n', '\n', ' ', ''])의 문자를 순서대로 사용하여 텍스트를 분할</font>하며, 분할된 청크들이 설정된 chunk_size보다 작아질 때까지 이 과정을 반복

   - <font color=red>split_documents()</font> : 데이터 로더의 반환값에서 page_content 항목을 찾아서 청크를 분리하는 함수
   - <font color=red>split_text()</font> : 데이터 로더의 반환값에서 page_content 항목만을 가져와서서 청크를 분리하는 함수
   - 반환값 구성
     - page_content : 분할된 토큰이 저장
     - metadata : 원본 문서에 대한 정보가 저장

In [ ]:
# 단순 구분자로 자르는 도구
from langchain_text_splitters import CharacterTextSplitter

# 재귀적으로 여러 구분자로 자른는 도구
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# 단순 chart splitter 생성
char_splitter = CharacterTextSplitter(
    separator = ' ',   # 구분자 설정
    chunk_size = 5,   # 기대하는 청크의 최대 사이즈
    chunk_overlap = 1   # 청크끼리 겹치는 구간 크기
)

In [ ]:
char_splitter.split_text('안녕하세요 오늘 점심은 무엇을 먹으면 좋을까요? 날씨가 화창하니 좋습니다.')

['안녕하세요', '오늘', '점심은', '무엇을', '먹으면', '좋을까요?', '날씨가', '화창하니', '좋습니다.']

In [ ]:
# Recursive splitter 사용하기
recursive_splitter = RecursiveCharacterTextSplitter(
    separators = [' ', ' '],   # 구분자 설정
    chunk_size = 10,   # 기대하는 청크의 최대 사이즈
    chunk_overlap = 5,   # 청크끼리 겹치는 구간크기
)

In [ ]:
recursive_splitter.split_text('안녕하세요 오늘 점심은 무엇을 먹으면 좋을까요? 날씨가 화창하니 좋습니다.')

['안녕하세요 오늘', '오늘 점심은', '점심은 무엇을', '무엇을 먹으면', '먹으면 좋을까요?', '날씨가 화창하니', '좋습니다.']

In [ ]:
kdt_splitter = RecursiveCharacterTextSplitter(
    separators = ['\n\n', '\n', ' ', ''],   # separators 생략시 문단, 줄바꿈, 띄어쓰기, 글자 순으로 자르도록 기본 설정 됨
    chunk_size = 500,   # 청크의 글자 크기
    chunk_overlap = 100   # 청크 오버랩 크기
)

In [ ]:
kdt_chunk = kdt_splitter.split_documents(documents)

In [ ]:
# 청크 사이즈 확인
for chunk in kdt_chunk :
    print(len(chunk.page_content))

132
469
5
497
457
379


In [ ]:
kdt_chunk[0]

Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2023-12-22T09:41:58+00:00', 'source': '/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM활용/data/KDT.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='- 3 -\n□ K-디지털 트레이닝 아카데미 유형\uf000디지털신기술아카데미:참여기업의훈련수요를기반으로참여기업과훈련기관간협약을체결한후설계된첨단산업·디지털분야훈련과정\uf000벤처\n스타트업아카데미:기업이필요로하는인재를양성하기위해벤처또는스타트업등의기업이속한협')

### (3) 임베딩 (Embedding) / 인덱싱 (Indexing)
 - 텍스트 데이터를 숫자로 이루어진 벡터로 변환하는 단계
 - 분할된 텍스트를 검색 가능한 형태로 만드는 단계

In [ ]:
# 임베딩 도구
from langchain_openai import OpenAIEmbeddings

# 벡터 DB 도구
from langchain_community.vectorstores import Chroma

In [ ]:
vec_db = Chroma.from_documents(
    documents = kdt_chunk,   # 임베딩할 벡터
    embedding = OpenAIEmbeddings()   # 임베딩 도구 연결
)

In [ ]:
vec_db

### (4) 검색 (Retrieval)
 - 사용자의 질문이나 주어진 컨텍스트에 가장 관련된 정보를 찾아내는 단계

In [ ]:
# 사용자 질문과 유사한 청크 2개를 찾는 검색기 생성
retriever = vec_db.as_retriever(search_kwargs = {'k' : 2})

In [ ]:
# retriever 실행 => runnable 요소
retriever.invoke('KDT 단가는 얼마야?')

[Document(metadata={'moddate': '2023-12-22T09:41:58+00:00', 'source': '/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM활용/data/KDT.pdf', 'producer': 'iLovePDF', 'creationdate': '', 'total_pages': 2, 'page_label': '2', 'page': 1, 'creator': 'PyPDF'}, page_content='에너지 7,686원 ②(18,150원*)첨단산업분야(12개직종)훈련과정및고성과맞춤형훈련과정인경우신청가능    * 훈련내용 및 훈련 수준 등을 검토하여 NCS단가 130%로 조정 가능-다만,첨단산업디지털선도기업은직종에관계없이단일단가(18,150원)신청가능 ③(NCS단가의300%이내또는훈련비실비*)3D프린팅,로봇,드론직종을제외한첨단산업분야(9개직종)훈련과정인경우신청가능    * NCS 단가의 300% 또는 훈련비 실비 과정은 기존 직업훈련시장에 없는 훈련과정이고 NCS 단가 130%, 18,150원 수준으로 운영이 어려운 과정에 한하여 예외적으로 인정함. 이 경우에도 심사과정에서 훈련내용 및 훈련 수준 등을 검토하여 18,150원 또는 NCS 단가 130%로 조정할 수 있음'),
 Document(metadata={'moddate': '2023-12-22T09:41:58+00:00', 'producer': 'iLovePDF', 'page': 1, 'source': '/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM활용/data/KDT.pdf', 'creator': 'PyPDF', 'creationdate': '', 'page_label': '2', 'total_pages': 2}, page_content='포함되지 않는 경우 예외적으로 허용하는 과정(전체 훈련시간의 30%이상 프로젝트학습으로 편성하여야 함)   ※ 신청 가능 아카데미: 첨

### (5) 생성 (Generation)
 - 검색된 정보를 바탕으로 사용자의 질문에 답변을 생성하는 최종 단계

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [ ]:
template = ChatPromptTemplate.from_messages([
    ('system', '당신은 직업훈련 과련 질문을 답변하는 챗봇입니다.'),
    ('system', '아래 Context를 참고해서 답변을 해주세요. 모르는 정보는 모른다고 솔직하게 이야기해주세요.'),
    ('system', 'context : {context}'),
    ('human', 'question : {question}')
])

In [ ]:
# 체인 구성하기
chatbot_chain = {'question' : RunnablePassthrough(),
                 'context' : retriever} | template | llm_4o_mini | StrOutputParser()

In [ ]:
print(chatbot_chain.invoke('KDT 단가는 얼마야?'))

KDT 단가는 특정 훈련 과정에 따라 다르며, 일반적으로는 NCS 단가의 130%로 조정된 금액을 기준으로 합니다. 특정 사례로는 다음과 같습니다:

- 신기술 훈련 과정의 경우, 분야별 평균 단가가 다양합니다:
  - 디지털: 7,303원
  - 소재·부품: 7,600원
  - 로봇·드론: 7,340원
  - 바이오헬스: 7,290원
  - 에코업: 7,568원
  - 에너지: 7,686원

- 첨단 산업 분야(12개 직종) 훈련 과정의 경우, 고성과 맞춤형 훈련 과정의 단가는 18,150원으로 설정되어 있습니다.

따라서, 특정 훈련 내용에 따라 다르게 적용됩니다.
